# Wedge：用 qust 标注楔形收敛结构

来源参考：[Investopedia](https://www.investopedia.com/terms/w/wedge.asp)


这篇 notebook 按 Investopedia 原页面的信息结构做完整中文改写，并把指标定义落成 `col(...).investopedia.xxx(...)` 的一行调用。能用 qust 现有 rolling、shift、select、with_cols、over 组合的就直接组合；需要 pivot/形态扫描的部分由 Rust helper 完成，Python 端不写 UDF。


## 1. Investopedia 原文内容完整改写：Wedge

### 什么是 Wedge
Wedge 是由两条逐渐收敛的趋势线构成的图表形态。价格仍在上下波动，但波动区间越来越窄。根据整体倾斜方向，可以分为 rising wedge 和 falling wedge。

### Rising Wedge
Rising wedge 中，高点和低点都在抬高，但低点抬升速度通常快于高点，导致区间收窄。虽然价格表面上还在上涨，但上行动能变弱，因此常被解释为潜在看跌结构。真正的确认通常来自跌破下边界。

### Falling Wedge
Falling wedge 中，高点和低点都在降低，但高点下降速度通常快于低点，区间同样收窄。它常被解释为潜在看涨结构，尤其在下跌末端出现时。确认通常来自向上突破上边界。

### 形态含义
楔形的重点不是方向本身，而是收敛。收敛说明价格每次扩展的空间变小，原趋势推动力可能下降。突破哪一侧，决定了形态是否按常见解释完成。

### 使用方式
交易者会画出上下趋势线，等待价格突破其中一条边界。也会结合成交量、趋势位置和更大周期结构判断。突破后，风险通常放在楔形内部或最近 swing 点附近。

### 局限性
楔形识别很主观：趋势线怎么画、需要几个触点、收敛程度多少才算有效，都没有唯一答案。程序化实现只能用近似方法，比如对 high/low 做线性拟合并检查区间是否缩小。楔形出现后也可能继续横盘或假突破。

## 2. 从文章到 qust 算子的落地

qust helper 在 lookback 窗口内对 high/low 拟合上下边界，检查区间是否收敛，并输出 `wedge_direction`：1 表示 rising wedge，-1 表示 falling wedge。`wedge_breakout` 标记常见确认方向是否已经突破。

## 3. qust 一行调用

```python
col("high", "low", "close").investopedia.wedge()
```

输入列顺序：`high, low, close`。

输出列：`wedge`, `wedge_direction`, `wedge_upper`, `wedge_lower`, `wedge_breakout`。

这些输出都保持和输入相同的行数，后面可以继续 `.with_cols(...)`、`.filter(...)`、`.monitor...`，也可以接 `.over("ticker", "ct")` 按合约独立计算。

In [1]:
import os
import sys

LOCAL_QUST_SOURCE = "/root/otters/otters-py/python"
if os.path.isdir(LOCAL_QUST_SOURCE) and LOCAL_QUST_SOURCE not in sys.path:
    sys.path.insert(0, LOCAL_QUST_SOURCE)

import qust as qs
import qust.future.future  # 注册 bt/stra/kline/fp 等金融命名空间
import qust.investopedia  # 注册 investopedia 命名空间
from qust import col, mark_shape
from qust._polars import pl

pl.Config.set_tbl_rows(16)
pl.Config.set_tbl_cols(28)

DATA_PATH = "/root/qust-py/examples/data/data_kline3.parquet"
PLOT_TICKER = "AP"


In [2]:
raw = pl.read_parquet(DATA_PATH).sort(["ticker", "ct", "datetime"])

base_contract = (
    raw
    .filter(pl.col("ticker") == PLOT_TICKER)
    .select("ct")
    .unique()
    .sort("ct")
    .get_column("ct")[0]
)

print("raw shape:", raw.shape)
print("tickers:", raw.select(pl.col("ticker").unique().sort()).to_series().to_list())
print("contract count:", raw.select("ticker", "ct").unique().height)
print("default plot ticker/ct:", PLOT_TICKER, base_contract)
raw.head(5)


raw shape: (408782, 8)
tickers: ['AP', 'RM', 'SA', 'al', 'eb', 'eg', 'fu', 'rb']
contract count: 141
default plot ticker/ct: AP 205


ticker,ct,datetime,open,high,low,close,volume
str,i32,datetime[ms],f64,f64,f64,f64,f64
"""AP""",205,2022-01-04 09:00:00,8394.0,8394.0,8392.0,8392.0,1100.0
"""AP""",205,2022-01-04 09:05:00,8385.0,8389.0,8348.0,8378.0,11169.0
"""AP""",205,2022-01-04 09:10:00,8375.0,8376.0,8298.0,8302.0,14001.0
"""AP""",205,2022-01-04 09:15:00,8301.0,8315.0,8271.0,8280.0,12839.0
"""AP""",205,2022-01-04 09:20:00,8279.0,8285.0,8243.0,8246.0,11496.0


## 4. 计算指标

下面用真实本地 K 线数据计算。对合约相关指标，示例都使用 `.over("ticker", "ct")`，表示每个品种、每个合约独立维护上下文，避免不同合约的数据串在一起。

In [3]:
indicator_expr = col("high", "low", "close").investopedia.wedge()
wedge_data = col.with_cols(indicator_expr).over("ticker", "ct").calc_data(raw)
plot_data = (
    wedge_data
    .filter((pl.col("ticker") == PLOT_TICKER) & (pl.col("ct") == base_contract))
    .sort("datetime")
    .head(1200)
)

summary = col(
    col("wedge").cast(pl.UInt32).sum().alias("wedge_count"),
    (col("wedge_direction") == col.lit(1)).cast(pl.UInt32).sum().alias("rising_wedge_count"),
    (col("wedge_direction") == col.lit(-1)).cast(pl.UInt32).sum().alias("falling_wedge_count"),
    col("wedge_breakout").cast(pl.UInt32).sum().alias("wedge_breakout_count"),
).calc_data(wedge_data)

print("plot shape:", plot_data.shape)
summary

plot shape: (1200, 13)


wedge_count,rising_wedge_count,falling_wedge_count,wedge_breakout_count
u32,u32,u32,u32
94327,48702,45625,21608


## 5. 用 monitor 画出来

图不是静态 PNG，而是 qust monitor 输出。你可以在 notebook 里放大、拖动、查看指标与 K 线的对应关系。

In [4]:
wedge_plot = col(
    col("datetime", "open", "high", "low", "close", "volume")
        .monitor("wedge_price", show_axis_label=True)
        .kline(),
    col("datetime", "wedge_upper", "wedge_lower")
        .monitor("wedge_price", show_axis_label=True)
        .line(),
    col("datetime", "close", "wedge")
        .monitor("wedge_price", show_axis_label=True)
        .mark(shape=mark_shape.circle, color="#4dd0e1", width=0.35),
    col("datetime", "close", "wedge_breakout")
        .monitor("wedge_price", show_axis_label=True)
        .mark(shape=mark_shape.triangle_up, color="#50fa7b", width=0.45),
).monitor.make_monitor("black").monitor.add_grid([
    ["wedge_price"],
]).runtime()

wedge_plot.plot(plot_data, open_in_jupyter=True, auto_open=False, height=560)

## 6. Wedge 策略回测

楔形方向取决于结构和突破。这里采用常见读法：下降楔形向上突破做多，上升楔形向下突破做空。`wedge_breakout` 必须成立才交易；入场后使用 3% 止盈、1.5% 止损，并将 `hold` 除以 `col.all.fp.vol_pms()`。

In [5]:
TAKE_PROFIT = 0.03
STOP_LOSS = 0.015


def make_two_sided_strategy(indicator_cols, open_long_raw, open_short_raw):
    """用当前指标生成完整多空策略；持仓用 fp.vol_pms 做品种/波动率尺度归一化。"""
    return (
        col
        .with_cols(indicator_cols)
        .with_cols(
            open_long_raw.fill_null(col.lit(False)).alias("open_long_raw"),
            open_short_raw.fill_null(col.lit(False)).alias("open_short_raw"),
        )
        # 指标在当前 K 线收盘后才确认，所以入场信号后移一根 K 线，避免同根 K 线偷看。
        .with_cols(
            col("open_long_raw").shift(1).expanding().fill_null(col.lit(False)).alias("open_long_sig"),
            col("open_short_raw").shift(1).expanding().fill_null(col.lit(False)).alias("open_short_sig"),
        )
        .with_cols(
            col("open_long_sig", "close").stra.exit_by_pct(TAKE_PROFIT, False).expanding().alias("take_profit_long"),
            col("open_long_sig", "close").stra.exit_by_pct(STOP_LOSS, True).expanding().alias("stop_loss_long"),
            col("open_short_sig", "close").stra.exit_by_pct(TAKE_PROFIT, True).expanding().alias("take_profit_short"),
            col("open_short_sig", "close").stra.exit_by_pct(STOP_LOSS, False).expanding().alias("stop_loss_short"),
        )
        .with_cols(
            (col("take_profit_long") | col("stop_loss_long") | col("open_short_sig"))
                .fill_null(col.lit(False))
                .alias("exit_long_sig"),
            (col("take_profit_short") | col("stop_loss_short") | col("open_long_sig"))
                .fill_null(col.lit(False))
                .alias("exit_short_sig"),
        )
        .with_cols(
            col("open_long_sig", "exit_long_sig", "open_short_sig", "exit_short_sig")
                .stra.to_hold_two_sides()
                .expanding()
                .alias("hold")
        )
        .with_cols(
            (col("hold") / col.all.fp.vol_pms()).alias("hold")
        )
        .with_cols(col("close", "hold").bt.price(fee_rate=0.0).expanding())
        .over("ticker", "ct")
        .select(
            col("pnl")
                .sum()
                .group_by(col("datetime").dt.date().alias("date"))
                .batch.sort("date")
                .with_cols(col("pnl").sum().expanding().alias("pnl_cum"))
                .select("date", "pnl", "pnl_cum")
        )
    )


def calc_strategy_stats(strategy_daily: pl.DataFrame) -> pl.DataFrame:
    return col(
        col("date").first_value().alias("start_date"),
        col("date").last_value().alias("end_date"),
        col.lit(1).sum().alias("days"),
        col("pnl").sum().alias("total_pnl"),
        col("pnl").mean().alias("mean_daily_pnl"),
        col("pnl").std().alias("std_daily_pnl"),
        (col("pnl").mean() / col("pnl").std() * col.lit(252 ** 0.5)).alias("sharpe_like"),
        col("pnl").min().alias("worst_day_pnl"),
        col("pnl").max().alias("best_day_pnl"),
    ).calc_data(strategy_daily)

indicator_cols = col("high", "low", "close").investopedia.wedge()
strategy_daily_expr = make_two_sided_strategy(
    indicator_cols,
    col("wedge_breakout") & (col("wedge_direction") == col.lit(-1)),
    col("wedge_breakout") & (col("wedge_direction") == col.lit(1)),
)
strategy_daily = strategy_daily_expr.calc_data(raw)
strategy_stats = calc_strategy_stats(strategy_daily)

print("strategy_daily shape:", strategy_daily.shape)
strategy_stats


strategy_daily shape: (859, 3)


start_date,end_date,days,total_pnl,mean_daily_pnl,std_daily_pnl,sharpe_like,worst_day_pnl,best_day_pnl
date,date,i32,f64,f64,f64,f64,f64,f64
2022-01-04,2024-12-31,859,19.555818,0.022926,1.586432,0.229406,-6.170336,7.326924


In [6]:
strategy_daily.tail(12)


date,pnl,pnl_cum
date,f64,f64
2024-12-18,0.305324,32.717111
2024-12-19,-3.687714,29.029397
2024-12-20,-0.147986,28.881412
2024-12-21,0.125345,29.006757
2024-12-23,-3.733066,25.273691
2024-12-24,-0.088078,25.185613
2024-12-25,-2.110001,23.075612
2024-12-26,0.170848,23.24646
2024-12-27,-2.024323,21.222137


## 7. 策略 PnL 曲线

下面用 qust monitor 同时画累计 PnL 和每日 PnL。累计曲线显示这套规则跨合约、跨日期后的整体资金变化；每日柱状图用来观察收益是否集中在少数日期。

In [7]:
pnl_dashboard = col(
    col("date", "pnl_cum")
        .monitor("strategy_pnl_cum", show_axis_label=True)
        .line(),
    col("date", "pnl")
        .monitor("strategy_daily_pnl", show_axis_label=True)
        .bar(),
).monitor.make_monitor("black").monitor.add_grid([
    ["strategy_pnl_cum"],
    ["strategy_daily_pnl"],
]).runtime()

pnl_dashboard.plot(strategy_daily, open_in_jupyter=True, auto_open=False, height=640)


## 8. 使用时的注意事项

- 技术指标只能把价格结构转成可计算规则，不等于确定性交易建议。
- 形态类指标通常需要后续 K 线确认；如果用于实时交易，应把确认延迟纳入回测。
- 参数越敏感，信号越多但噪声越大；参数越保守，信号更少但滞后更明显。
- 在多合约或多股票数据上使用时，优先写 `.over("ticker", "ct")` 或合适的分组键。